In [ ]:
import numpy as np
from pathlib import Path
import sys

PROJECT_DIR = str(Path.cwd().parent) + "/"
sys.path.append(PROJECT_DIR)

from mpp_project.end_game_solver import solve_endgame_dp
# Solveur théorique exact partagé avec la suite pytest (tests/helpers.py)
from tests.helpers import exact_theoretical_wr, terminal_ge_zero

print("🌌 DÉMARRAGE DU LABORATOIRE DP : PROFONDEUR TEMPORELLE (4 MATCHS)")

# ==========================================
# 1. CRÉATION DES 4 MATCHS "PIPEAU" (t=28 à 31)
# ==========================================
N_MATCHS = 4
stop_t = 32 - N_MATCHS # 28

true_proba = np.array([0.60, 0.25, 0.15], dtype=np.float32) 
crowd = np.array([0.70, 0.20, 0.10], dtype=np.float32)
# On choisit des gains qui donnent des écarts pairs (pour éviter les arrondis dans la grille / 2.0)
gains = np.array([20, 50, 90], dtype=np.int32) 

match_probs = np.zeros((32, 3), dtype=np.float32)
crowds = np.zeros((32, 3), dtype=np.float32)
gains_1N2 = np.zeros((32, 3), dtype=np.int32)

for t in range(stop_t, 32):
    match_probs[t] = true_proba
    crowds[t] = crowd
    gains_1N2[t] = gains

p_empirique_1D = np.zeros((32, 3, 250), dtype=np.float32)
p_empirique_1D[:, :, 0] = 1.0 # Peloton désactivé (fantôme)

alphas = np.ones(32, dtype=np.float32)
roles_mock = np.full(32, -1, dtype=np.int32)

V_term = np.zeros((501, 501, 2, 2, 2, 2), dtype=np.float32)
for g1 in range(501):
    for g2 in range(501):
        if (g1 - 250) >= 0 and (g2 - 250) >= 0:
            V_term[g1, g2, :, :, :, :] = 1.0

# ==========================================
# 2. LE SOLVEUR THÉORIQUE EXACT (importé de tests/helpers.py)
# ==========================================
# exact_theoretical_wr(gap, n_matchs, true_proba, crowd, gains, has_booster, terminal)
# énumère tout l'arbre combinatoire. Terminal `gap >= 0 -> victoire`, aligné sur V_term.

# ==========================================
# 3. EXÉCUTION DE L'ORACLE NUMBA (DP)
# ==========================================
print(f"🧠 Lancement de la DP (Sur {N_MATCHS} matchs)...")
V_history = solve_endgame_dp(
    match_probs, crowds, gains_1N2, p_empirique_1D, alphas, 
    roles_mock, roles_mock, roles_mock, V_term, stop_t=stop_t
)

# On extrait la décision à t=28 (Il reste 4 matchs)
Q_table_t28 = V_history[stop_t]

# ==========================================
# 4. LES TESTS (Confrontation des deux mondes)
# ==========================================
print("\n" + "="*50)
print(f"📊 RÉSULTATS DES TESTS (Horizon = {N_MATCHS} matchs)")
print("="*50)

idx_g2_safe = 500  

def test_scenario_multiple(nom, gap_initial):
    # Calcul DP (Lecture instantanée dans la grille générée)
    gap_grid = int(gap_initial / 2.0)
    idx_g1 = gap_grid + 250
    wr_dp = Q_table_t28[idx_g1, idx_g2_safe, 0, 0, 0, 0]
    
    # Calcul Théorique (Parcours de l'arbre combinatoire via le helper)
    wr_th = exact_theoretical_wr(gap_initial, N_MATCHS, true_proba, crowd, gains,
                                 has_booster=False, terminal=terminal_ge_zero)
    
    print(f"▶️ TEST : {nom}")
    print(f"   Retard sur Bob : {-gap_initial} pts")
    print(f"   WR Théorique   : {wr_th*100:05.2f}%")
    print(f"   WR Calculé DP  : {wr_dp*100:05.2f}%")
    
    if abs(wr_dp - wr_th) < 0.001:
        print("   ✅ TEST RÉUSSI")
    else:
        print("   ❌ ÉCHEC DU TEST")
    print("-" * 50)


max_gain_possible = N_MATCHS * 90 # 4 * 90 = 360

# --- SCÉNARIO 1 : MISSION IMPOSSIBLE ---
test_scenario_multiple("Mission Impossible absolue", gap_initial=-(max_gain_possible + 20))

# --- SCÉNARIO 2 : LE GRAND CHELEM OBLIGATOIRE ---
test_scenario_multiple("Le Grand Chelem (4 Outsiders Requis)", gap_initial=-350)

# --- SCÉNARIO 3 : LE DROIT À L'ERREUR ---
test_scenario_multiple("Le Droit à l'Erreur (3 Outsiders + 1 Nul suffisent)", gap_initial=-300)

# --- SCÉNARIO 4 : L'OUTSIDER DÈS LE PREMIER MATCH ? ---
test_scenario_multiple("Retard modéré (Stratégie Mixte)", gap_initial=-150)

# --- SCÉNARIO 5 : ON GÈRE L'AVANCE ---
test_scenario_multiple("Gestion de l'avance pendant 4 matchs", gap_initial=50)

In [ ]:
print("🚀 DÉMARRAGE DU LABORATOIRE DP : PROFONDEUR TEMPORELLE + BOOSTER x2")

# ==========================================
# 1. CRÉATION DES 4 MATCHS "PIPEAU" (t=28 à 31)
# ==========================================
N_MATCHS = 4
stop_t = 32 - N_MATCHS # 28

true_proba = np.array([0.60, 0.25, 0.15], dtype=np.float32) 
crowd = np.array([0.70, 0.20, 0.10], dtype=np.float32)
gains = np.array([20, 50, 90], dtype=np.int32) 

match_probs = np.zeros((32, 3), dtype=np.float32)
crowds = np.zeros((32, 3), dtype=np.float32)
gains_1N2 = np.zeros((32, 3), dtype=np.int32)

for t in range(stop_t, 32):
    match_probs[t] = true_proba
    crowds[t] = crowd
    gains_1N2[t] = gains

p_empirique_1D = np.zeros((32, 3, 250), dtype=np.float32)
p_empirique_1D[:, :, 0] = 1.0 # Peloton désactivé (fantôme)

alphas = np.ones(32, dtype=np.float32)
roles_mock = np.full(32, -1, dtype=np.int32)

V_term = np.zeros((501, 501, 2, 2, 2, 2), dtype=np.float32)
for g1 in range(501):
    for g2 in range(501):
        if (g1 - 250) >= 0 and (g2 - 250) >= 0:
            V_term[g1, g2, :, :, :, :] = 1.0

# ==========================================
# 2. LE SOLVEUR THÉORIQUE EXACT (importé de tests/helpers.py)
# ==========================================
# Le booster se gère via l'argument has_booster=True (l'arbre teste à chaque
# match de jouer ou non le x2, en propageant le booster restant).

# ==========================================
# 3. EXÉCUTION DE L'ORACLE NUMBA (DP)
# ==========================================
print(f"🧠 Lancement de la DP (Sur {N_MATCHS} matchs)...")
V_history = solve_endgame_dp(
    match_probs, crowds, gains_1N2, p_empirique_1D, alphas, 
    roles_mock, roles_mock, roles_mock, V_term, stop_t=stop_t
)

# On extrait la décision à t=28
Q_table_t28 = V_history[stop_t]

# ==========================================
# 4. LES TESTS (Confrontation des deux mondes)
# ==========================================
print("\n" + "="*50)
print(f"📊 RÉSULTATS DES TESTS (Horizon = {N_MATCHS} matchs | Base vs x2)")
print("="*50)

idx_g2_safe = 500  

def test_scenario_booster_multiple(nom, gap_initial):
    gap_grid = int(gap_initial / 2.0)
    idx_g1 = gap_grid + 250
    
    # --- SANS BOOSTER (Dimension 0) ---
    wr_dp_sans = Q_table_t28[idx_g1, idx_g2_safe, 0, 0, 0, 0]
    wr_th_sans = exact_theoretical_wr(gap_initial, N_MATCHS, true_proba, crowd, gains,
                                      has_booster=False, terminal=terminal_ge_zero)
    
    # --- AVEC BOOSTER (Dimension 1) ---
    wr_dp_avec = Q_table_t28[idx_g1, idx_g2_safe, 1, 0, 0, 0]
    wr_th_avec = exact_theoretical_wr(gap_initial, N_MATCHS, true_proba, crowd, gains,
                                      has_booster=True, terminal=terminal_ge_zero)
    
    print(f"▶️ TEST : {nom}")
    print(f"   Retard sur Bob : {-gap_initial} pts")
    print(f"   [Base] Théorie: {wr_th_sans*100:05.2f}% | DP: {wr_dp_sans*100:05.2f}%")
    print(f"   [ x2 ] Théorie: {wr_th_avec*100:05.2f}% | DP: {wr_dp_avec*100:05.2f}%")
    
    reussi = abs(wr_dp_sans - wr_th_sans) < 0.001 and abs(wr_dp_avec - wr_th_avec) < 0.001
    if reussi:
        print("   ✅ TEST RÉUSSI")
    else:
        print("   ❌ ÉCHEC DU TEST")
    print("-" * 50)


# Gain max base 4 matchs : 4*90 = 360 ; avec booster : 3*90 + 180 = 450

# --- SCÉNARIO 1 : MISSION IMPOSSIBLE (MÊME AVEC BOOSTER) ---
test_scenario_booster_multiple("Désespoir absolu", gap_initial=-460)

# --- SCÉNARIO 2 : LE BOOSTER EST LA SEULE ISSUE ---
test_scenario_booster_multiple("Le Booster de la dernière chance", gap_initial=-400)

# --- SCÉNARIO 3 : LE DROIT À L'ERREUR ÉLARGI ---
test_scenario_booster_multiple("Le Droit à l'erreur (Boost de probabilité)", gap_initial=-300)

# --- SCÉNARIO 4 : L'OPTIMISATION DU RISQUE ---
test_scenario_booster_multiple("Gestion stratégique du retard", gap_initial=-100)

# --- SCÉNARIO 5 : LE LEVIER SÉCURITAIRE ---
test_scenario_booster_multiple("L'Estocade du Leader", gap_initial=50)